📚 Part 1: What is a Knowledge Graph?
Definition
A Knowledge Graph is a structured representation of information that captures entities (nodes), their attributes (properties), and the relationships between them (edges). Unlike traditional databases that store data in tables, knowledge graphs store data as interconnected networks that mirror how information naturally relates in the real world.
Simple Analogy:

Relational Database: Like a filing cabinet with folders and forms
Knowledge Graph: Like a mind map or social network showing how everything connects


Nodes (Entities): Represent objects or concepts

Examples: Programs, Files, Procedures, Variables


Edges (Relationships): Define how nodes connect

Examples: CALLS, READS, WRITES, CONTAINS


Properties (Attributes): Store information about nodes and edges

Examples: name, description, line_count, created_date

Why Knowledge Graphs Matter for Legacy Code
Traditional documentation approaches:

📄 Static Documentation: Becomes outdated quickly
📊 Spreadsheets: Can't capture complex relationships
🗂️ Relational Databases: Requires complex JOINs for multi-hop queries
💬 Tribal Knowledge: Lost when experts retire

Knowledge Graphs solve these by:

✅ Living Documentation: Updates with code changes
✅ Relationship-First: Queries traverse connections naturally
✅ Visual Understanding: Developers see the big picture
✅ Pattern Discovery: AI can identify architectural patterns
✅ Impact Analysis: Instant "what-if" scenarios

How Knowledge Graphs Help
1. Code Understanding
Transform unreadable COBOL into queryable knowledge:
Before (COBOL Code):
  CALL 'CUSTUPDT' USING CUSTOMER-RECORD TRANSACTION-RECORD.
  
After (Knowledge Graph):
  (CUSTMAST)-[:CALLS {params: "CUSTOMER-RECORD, TRANSACTION-RECORD"}]->(CUSTUPDT)
2. Impact Analysis
Answer critical questions in seconds:
Question: "What breaks if I change the CUSTOMER-FILE format?"
Traditional Approach:

Grep through thousands of files
Manual code review (weeks)
Interview old-timers
Cross-reference documentation (if it exists)
Result: 3-4 weeks, still uncertain

Knowledge Graph Approach:
cypherMATCH (f:CobolFile {name: 'CUSTOMER-FILE'})
      <-[:READS|WRITES]-(p:CobolProgram)
RETURN p.name, p.description

Result: 2 seconds, complete list

Part 3: Benefits of Knowledge Graphs
Technical Benefits
BenefitDescriptionImpactMulti-hop QueriesFind indirect dependencies in single query10x faster analysisGraph AlgorithmsPageRank, Community Detection, Shortest PathDiscover hidden insightsFlexible SchemaAdd new relationship types without migrationAdapt as you learnVisual ExplorationSee code structure graphicallyFaster onboardingPattern MatchingFind similar code structuresReuse modernization strategies
Business Benefits

Risk Reduction (60-80%)

Know exact blast radius before changes
Test only what's actually affected
Prevent production incidents


Time Savings (50-70%)

Impact analysis: weeks → minutes
Code understanding: months → days
Modernization planning: manual → automated


Knowledge Preservation

Captures expert knowledge in graph
Survives employee turnover
Searchable institutional memory


Better Decision Making

Data-driven modernization priorities
ROI calculation for each module
Resource allocation optimization


Gradual Modernization

Identify independent modules
Minimize disruption
Validate each step



Comparison: Traditional vs. Knowledge Graph Approach
AspectTraditional ApproachKnowledge Graph ApproachImpact Analysis2-4 weeks manual review2 seconds queryCode UnderstandingRead thousands of linesQuery relationshipsDocumentationStatic, outdated PDFsLiving, queryable graphDependency TrackingSpreadsheets, grepAutomated graph traversalOnboarding New Devs3-6 months2-4 weeksModernization PlanningGut feel, politicsData-driven, objectiveRisk AssessmentSubjective estimatesQuantified blast radius

    ┌──────────────┐
    │  Keanu       │
    │  Reeves      │
    └──────┬───────┘
           │
     [ACTED_IN]
     role: Neo
           │
           ↓
    ┌──────────────┐         ┌──────────────┐
    │  The Matrix  │←────────│    Lana      │
    │    (1999)    │ DIRECTED│  Wachowski   │
    └──────────────┘         └──────────────┘
    
    
    ┌──────────────┐         ┌──────────────┐
    │  Leonardo    │         │ Christopher  │
    │  DiCaprio    │         │    Nolan     │
    └──────┬───────┘         └──────┬───────┘
           │                        │
     [ACTED_IN]              [DIRECTED]
     role: Cobb                     │
           │                        │
           ↓                        ↓
    ┌──────────────┐         ┌──────────────┐
    │  Inception   │←────────┤ The Dark     │
    │    (2010)    │         │   Knight     │
    └──────────────┘         │    (2008)    │
                             └──────▲───────┘
                                    │
                              [ACTED_IN]
                              role: Batman
                                    │
                             ┌──────┴───────┐
                             │  Christian   │
                             │    Bale      │
                             └──────────────┘

In [85]:
!pip install langchain langchain-openai langchain-neo4j langchain-community python-dotenv langchain_groq

In [86]:
import os
import re
from typing import List, Dict, Any
from dotenv import load_dotenv

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain, Neo4jVector
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Load environment variables
load_dotenv()

print("✓ Imports loaded successfully")

✓ Imports loaded successfully


In [87]:
from google.colab import userdata
import os

# Load secrets from Colab's secret manager
os.environ["NEO4J_URI"] = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✓ Neo4j,OPENAI_API_KEY and GROQ_API_KEY credentials loaded from Colab secrets")

✓ Neo4j,OPENAI_API_KEY and GROQ_API_KEY credentials loaded from Colab secrets


In [88]:
# Initialize Neo4j connection
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD")
)

# Initialize OpenAI LLM
llm = ChatOpenAI(temperature=0, model="gpt-4o",api_key=os.getenv("OPENAI_API_KEY"))

#Initialize Groq Model
llm_groq = ChatGroq(temperature=0, model="llama-3.3-70b-versatile",api_key=os.getenv("GROQ_API_KEY"))

print("✓ Neo4j,OpenAI and Groq connections established")
print(f"Neo4j URI: {os.getenv('NEO4J_URI')}")

✓ Neo4j,OpenAI and Groq connections established
Neo4j URI: neo4j+s://4c3fce54.databases.neo4j.io


In [89]:
output=llm_groq.invoke("What is the capital of France?")
output.content

result=llm.invoke("What is the capital of India?")
result.content

print(output.content)
print(result.content)

The capital of France is Paris.
The capital of India is New Delhi.


In [90]:
# Clear existing COBOL data (optional - for fresh start)
graph.query("""
MATCH (n)
WHERE n:CobolProgram OR n:Procedure OR n:DataFile OR n:Variable OR n:Copybook
DETACH DELETE n
""")
print("✓ Cleared existing COBOL data from graph")

✓ Cleared existing COBOL data from graph


## 2. Sample COBOL Code

We'll create realistic COBOL programs that demonstrate:
- Program calls and dependencies
- File operations (READ/WRITE)
- Business logic
- Data structures

In [91]:
SAMPLE_COBOL_PROGRAMS = {
    "CUSTMAST": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. CUSTMAST.
       AUTHOR. LEGACY-TEAM.
       DATE-WRITTEN. 1985-03-15.
      *****************************************************************
      * CUSTOMER MASTER FILE MAINTENANCE PROGRAM                     *
      * READS CUSTOMER TRANSACTIONS AND UPDATES MASTER FILE          *
      *****************************************************************

       ENVIRONMENT DIVISION.
       INPUT-OUTPUT SECTION.
       FILE-CONTROL.
           SELECT CUSTOMER-FILE ASSIGN TO CUSTFILE
               ORGANIZATION IS INDEXED
               ACCESS MODE IS RANDOM
               RECORD KEY IS CUST-ID.

           SELECT TRANS-FILE ASSIGN TO TRANSFILE
               ORGANIZATION IS SEQUENTIAL.

       DATA DIVISION.
       FILE SECTION.
       FD  CUSTOMER-FILE.
       01  CUSTOMER-RECORD.
           05  CUST-ID              PIC 9(8).
           05  CUST-NAME            PIC X(50).
           05  CUST-ADDRESS         PIC X(100).
           05  CUST-BALANCE         PIC 9(9)V99.
           05  CUST-STATUS          PIC X.

       FD  TRANS-FILE.
       01  TRANSACTION-RECORD.
           05  TRANS-CUST-ID        PIC 9(8).
           05  TRANS-TYPE           PIC X.
           05  TRANS-AMOUNT         PIC 9(7)V99.

       WORKING-STORAGE SECTION.
       01  WS-EOF                   PIC X VALUE 'N'.
       01  WS-CUSTOMER-COUNT        PIC 9(5) VALUE 0.

       PROCEDURE DIVISION.
       MAIN-PROCESS.
           PERFORM INITIALIZE-FILES
           PERFORM PROCESS-TRANSACTIONS UNTIL WS-EOF = 'Y'
           PERFORM FINALIZE-PROCESSING
           STOP RUN.

       INITIALIZE-FILES.
           OPEN INPUT TRANS-FILE
           OPEN I-O CUSTOMER-FILE.

       PROCESS-TRANSACTIONS.
           READ TRANS-FILE
               AT END MOVE 'Y' TO WS-EOF
               NOT AT END
                   PERFORM UPDATE-CUSTOMER
           END-READ.

       UPDATE-CUSTOMER.
           MOVE TRANS-CUST-ID TO CUST-ID
           READ CUSTOMER-FILE
               INVALID KEY
                   CALL 'CUSTADD' USING TRANSACTION-RECORD
               NOT INVALID KEY
                   CALL 'CUSTUPDT' USING CUSTOMER-RECORD TRANSACTION-RECORD
           END-READ
           CALL 'AUDITLOG' USING TRANSACTION-RECORD.

       FINALIZE-PROCESSING.
           CLOSE TRANS-FILE
           CLOSE CUSTOMER-FILE
           DISPLAY 'PROCESSING COMPLETE. RECORDS: ' WS-CUSTOMER-COUNT.
""",

    "CUSTUPDT": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. CUSTUPDT.
       AUTHOR. LEGACY-TEAM.
      *****************************************************************
      * CUSTOMER UPDATE SUBPROGRAM                                   *
      * APPLIES TRANSACTION TO CUSTOMER RECORD                       *
      *****************************************************************

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-NEW-BALANCE           PIC 9(9)V99.

       LINKAGE SECTION.
       01  LS-CUSTOMER-RECORD.
           05  LS-CUST-ID           PIC 9(8).
           05  LS-CUST-NAME         PIC X(50).
           05  LS-CUST-BALANCE      PIC 9(9)V99.

       01  LS-TRANSACTION.
           05  LS-TRANS-TYPE        PIC X.
           05  LS-TRANS-AMOUNT      PIC 9(7)V99.

       PROCEDURE DIVISION USING LS-CUSTOMER-RECORD LS-TRANSACTION.
       MAIN-LOGIC.
           EVALUATE LS-TRANS-TYPE
               WHEN 'D'
                   PERFORM PROCESS-DEBIT
               WHEN 'C'
                   PERFORM PROCESS-CREDIT
               WHEN OTHER
                   DISPLAY 'INVALID TRANSACTION TYPE'
           END-EVALUATE
           CALL 'VALIDATE' USING LS-CUSTOMER-RECORD
           GOBACK.

       PROCESS-DEBIT.
           SUBTRACT LS-TRANS-AMOUNT FROM LS-CUST-BALANCE.

       PROCESS-CREDIT.
           ADD LS-TRANS-AMOUNT TO LS-CUST-BALANCE.
""",

    "CUSTADD": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. CUSTADD.
       AUTHOR. LEGACY-TEAM.
      *****************************************************************
      * CUSTOMER ADD SUBPROGRAM                                      *
      * CREATES NEW CUSTOMER RECORDS                                 *
      *****************************************************************

       ENVIRONMENT DIVISION.
       INPUT-OUTPUT SECTION.
       FILE-CONTROL.
           SELECT CUSTOMER-FILE ASSIGN TO CUSTFILE
               ORGANIZATION IS INDEXED
               ACCESS MODE IS RANDOM
               RECORD KEY IS CUST-ID.

       DATA DIVISION.
       FILE SECTION.
       FD  CUSTOMER-FILE.
       01  CUSTOMER-RECORD.
           05  CUST-ID              PIC 9(8).
           05  CUST-NAME            PIC X(50).
           05  CUST-BALANCE         PIC 9(9)V99.

       LINKAGE SECTION.
       01  LS-TRANSACTION.
           05  LS-TRANS-CUST-ID     PIC 9(8).
           05  LS-TRANS-TYPE        PIC X.
           05  LS-TRANS-AMOUNT      PIC 9(7)V99.

       PROCEDURE DIVISION USING LS-TRANSACTION.
       MAIN-LOGIC.
           PERFORM CREATE-CUSTOMER
           CALL 'AUDITLOG' USING LS-TRANSACTION
           GOBACK.

       CREATE-CUSTOMER.
           MOVE LS-TRANS-CUST-ID TO CUST-ID
           MOVE SPACES TO CUST-NAME
           MOVE LS-TRANS-AMOUNT TO CUST-BALANCE
           WRITE CUSTOMER-RECORD.
""",

    "AUDITLOG": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. AUDITLOG.
      *****************************************************************
      * AUDIT LOGGING UTILITY                                        *
      * WRITES TRANSACTION AUDIT TRAIL                               *
      *****************************************************************

       ENVIRONMENT DIVISION.
       INPUT-OUTPUT SECTION.
       FILE-CONTROL.
           SELECT AUDIT-FILE ASSIGN TO AUDITFILE
               ORGANIZATION IS SEQUENTIAL.

       DATA DIVISION.
       FILE SECTION.
       FD  AUDIT-FILE.
       01  AUDIT-RECORD             PIC X(200).

       LINKAGE SECTION.
       01  LS-TRANSACTION           PIC X(100).

       PROCEDURE DIVISION USING LS-TRANSACTION.
       MAIN-LOGIC.
           OPEN EXTEND AUDIT-FILE
           WRITE AUDIT-RECORD FROM LS-TRANSACTION
           CLOSE AUDIT-FILE
           GOBACK.
""",

    "VALIDATE": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. VALIDATE.
      *****************************************************************
      * CUSTOMER VALIDATION UTILITY                                  *
      * VALIDATES CUSTOMER DATA INTEGRITY                            *
      *****************************************************************

       LINKAGE SECTION.
       01  LS-CUSTOMER-RECORD.
           05  LS-CUST-ID           PIC 9(8).
           05  LS-CUST-BALANCE      PIC 9(9)V99.

       PROCEDURE DIVISION USING LS-CUSTOMER-RECORD.
       MAIN-LOGIC.
           IF LS-CUST-BALANCE < 0
               DISPLAY 'WARNING: NEGATIVE BALANCE FOR ' LS-CUST-ID
           END-IF
           GOBACK.
""",
    "REPORTGEN": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. REPORTGEN.
       AUTHOR. LEGACY-ANALYST.
      *****************************************************************
      * CUSTOMER REPORT GENERATION PROGRAM                           *
      * READS CUSTOMER MASTER FILE AND GENERATES A SUMMARY REPORT    *
      *****************************************************************

       ENVIRONMENT DIVISION.
       INPUT-OUTPUT SECTION.
       FILE-CONTROL.
           SELECT CUSTOMER-FILE ASSIGN TO CUSTFILE
               ORGANIZATION IS INDEXED
               ACCESS MODE IS SEQUENTIAL.
           SELECT REPORT-FILE ASSIGN TO RPTFILE
               ORGANIZATION IS SEQUENTIAL.

       DATA DIVISION.
       FILE SECTION.
       FD  CUSTOMER-FILE.
       01  CUSTOMER-RECORD-IN.
           05  CUST-ID-IN           PIC 9(8).
           05  CUST-NAME-IN         PIC X(50).
           05  CUST-BALANCE-IN      PIC 9(9)V99.

       FD  REPORT-FILE.
       01  REPORT-LINE              PIC X(132).

       WORKING-STORAGE SECTION.
       01  WS-EOF-CUST              PIC X VALUE 'N'.
       01  WS-TOTAL-BALANCE         PIC 9(12)V99 VALUE 0.

       PROCEDURE DIVISION.
       MAIN-REPORT-PROCESS.
           OPEN INPUT CUSTOMER-FILE
           OPEN OUTPUT REPORT-FILE
           PERFORM READ-CUSTOMER-FILE
           PERFORM PROCESS-CUSTOMER-RECORDS UNTIL WS-EOF-CUST = 'Y'
           PERFORM WRITE-SUMMARY-REPORT
           CLOSE CUSTOMER-FILE
           CLOSE REPORT-FILE
           CALL 'AUDITLOG' USING 'REPORTGEN COMPLETED'
           GOBACK.

       READ-CUSTOMER-FILE.
           READ CUSTOMER-FILE NEXT RECORD
               AT END MOVE 'Y' TO WS-EOF-CUST
           END-READ.

       PROCESS-CUSTOMER-RECORDS.
           ADD CUST-BALANCE-IN TO WS-TOTAL-BALANCE
           MOVE CUST-ID-IN TO REPORT-LINE(1:8)
           MOVE CUST-NAME-IN TO REPORT-LINE(10:50)
           MOVE CUST-BALANCE-IN TO REPORT-LINE(65:12)
           WRITE REPORT-LINE
           PERFORM READ-CUSTOMER-FILE.

       WRITE-SUMMARY-REPORT.
           MOVE 'TOTAL BALANCE: ' TO REPORT-LINE(1:15)
           MOVE WS-TOTAL-BALANCE TO REPORT-LINE(20:15)
           WRITE REPORT-LINE.
""",
    "DAILYPROC": """
       IDENTIFICATION DIVISION.
       PROGRAM-ID. DAILYPROC.
       AUTHOR. LEGACY-SCHEDULER.
      *****************************************************************
      * DAILY BATCH PROCESSING PROGRAM                               *
      * COORDINATES EXECUTION OF VARIOUS DAILY JOBS                  *
      *****************************************************************

       ENVIRONMENT DIVISION.
       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-DATE                  PIC 9(8).

       PROCEDURE DIVISION.
       MAIN-DAILY-JOB.
           ACCEPT WS-DATE FROM DATE YYYYMMDD.
           DISPLAY 'STARTING DAILY PROCESSING FOR ' WS-DATE.

           CALL 'CUSTMAST'.
           CALL 'REPORTGEN'.
           CALL 'AUDITLOG' USING 'DAILYPROC FINISHED'.

           DISPLAY 'DAILY PROCESSING COMPLETE.'.
           GOBACK.
"""
}

print(f"✓ Loaded {len(SAMPLE_COBOL_PROGRAMS)} sample COBOL programs")
for prog_name in SAMPLE_COBOL_PROGRAMS.keys():
    print(f"  - {prog_name}")

✓ Loaded 7 sample COBOL programs
  - CUSTMAST
  - CUSTUPDT
  - CUSTADD
  - AUDITLOG
  - VALIDATE
  - REPORTGEN
  - DAILYPROC


## 3. COBOL Parser

Extract key information from COBOL code:
- Program names and metadata
- CALL statements (dependencies)
- File operations (READ/WRITE)
- Procedures/Paragraphs
- Variables and data structures

In [92]:
class CobolParser:
    """Simple regex-based COBOL parser for MVP."""

    def __init__(self):
        # Regex patterns for COBOL constructs
        self.patterns = {
            'program_id': r'PROGRAM-ID\.\s+(\w+)',
            'author': r'AUTHOR\.\s+(.+)',
            'call': r'CALL\s+[\'"](\w+)[\'"]',
            'file_read': r'READ\s+(\w+)',
            'file_write': r'WRITE\s+(\w+)',
            'file_open': r'OPEN\s+(?:INPUT|OUTPUT|I-O|EXTEND)\s+(\w+)',
            'procedure': r'^\s{7}(\w+-\w+(?:-\w+)*)\.',
            'variable': r'^\s+\d+\s+(\w+-\w+(?:-\w+)*)\s+PIC',
            'file_select': r'SELECT\s+(\w+)\s+ASSIGN',
        }

    def parse_program(self, program_name: str, code: str) -> Dict[str, Any]:
        """Parse a COBOL program and extract metadata."""

        result = {
            'program_name': program_name,
            'author': None,
            'calls': [],
            'files_read': set(),
            'files_written': set(),
            'files_declared': set(),
            'procedures': [],
            'variables': [],
            'description': self._extract_description(code),
            'code': code
        }

        # Extract program ID
        prog_match = re.search(self.patterns['program_id'], code, re.IGNORECASE)
        if prog_match:
            result['program_name'] = prog_match.group(1)

        # Extract author
        author_match = re.search(self.patterns['author'], code, re.IGNORECASE)
        if author_match:
            result['author'] = author_match.group(1).strip()

        # Extract CALL statements
        result['calls'] = re.findall(self.patterns['call'], code, re.IGNORECASE)

        # Extract file operations
        result['files_read'] = set(re.findall(self.patterns['file_read'], code, re.IGNORECASE))
        result['files_written'] = set(re.findall(self.patterns['file_write'], code, re.IGNORECASE))
        result['files_declared'] = set(re.findall(self.patterns['file_select'], code, re.IGNORECASE))

        # Extract procedures
        result['procedures'] = re.findall(self.patterns['procedure'], code, re.MULTILINE | re.IGNORECASE)

        # Extract variables (limit to first 20 to avoid clutter)
        result['variables'] = re.findall(self.patterns['variable'], code, re.MULTILINE | re.IGNORECASE)[:20]

        return result

    def _extract_description(self, code: str) -> str:
        """Extract description from comments."""
        comment_lines = []
        for line in code.split('\n'):
            if line.strip().startswith('*') and len(line.strip()) > 1:
                comment_lines.append(line.strip()[1:].strip())

        # Return first few meaningful comment lines
        meaningful = [c for c in comment_lines if len(c) > 10 and not c.startswith('*')]
        return ' '.join(meaningful[:3]) if meaningful else ''

    def parse_all(self, programs: Dict[str, str]) -> List[Dict[str, Any]]:
        """Parse all COBOL programs."""
        return [self.parse_program(name, code) for name, code in programs.items()]

# Test the parser
parser = CobolParser()
parsed_programs = parser.parse_all(SAMPLE_COBOL_PROGRAMS)

print(f"✓ Parsed {len(parsed_programs)} COBOL programs\n")
for prog in parsed_programs:
    print(f"Program: {prog['program_name']}")
    print(f"  Calls: {prog['calls']}")
    print(f"  Files Read: {prog['files_read']}")
    print(f"  Files Written: {prog['files_written']}")
    print(f"  Procedures: {len(prog['procedures'])}")
    print()

✓ Parsed 7 COBOL programs

Program: CUSTMAST
  Calls: ['CUSTADD', 'CUSTUPDT', 'AUDITLOG']
  Files Read: {'CUSTOMER', 'CALL', 'TRANS'}
  Files Written: set()
  Procedures: 8

Program: CUSTUPDT
  Calls: ['VALIDATE']
  Files Read: set()
  Files Written: set()
  Procedures: 4

Program: CUSTADD
  Calls: ['AUDITLOG']
  Files Read: set()
  Files Written: {'CUSTOMER'}
  Procedures: 4

Program: AUDITLOG
  Calls: []
  Files Read: set()
  Files Written: {'AUDIT'}
  Procedures: 3

Program: VALIDATE
  Calls: []
  Files Read: set()
  Files Written: set()
  Procedures: 2

Program: REPORTGEN
  Calls: ['AUDITLOG']
  Files Read: {'CUSTOMER'}
  Files Written: {'REPORT'}
  Procedures: 6

Program: DAILYPROC
  Calls: ['CUSTMAST', 'REPORTGEN', 'AUDITLOG']
  Files Read: set()
  Files Written: set()
  Procedures: 2



## 4. Knowledge Graph Builder

Convert parsed COBOL data into Neo4j knowledge graph with proper relationships.

In [93]:
class CobolKnowledgeGraphBuilder:
    """Build Neo4j knowledge graph from parsed COBOL programs."""

    def __init__(self, graph: Neo4jGraph):
        self.graph = graph

    def build_graph(self, parsed_programs: List[Dict[str, Any]]):
        """Build complete knowledge graph from parsed programs."""

        print("Building knowledge graph...")

        for prog in parsed_programs:
            # Create program node
            self._create_program_node(prog)

            # Create file nodes and relationships
            self._create_file_relationships(prog)

            # Create procedure nodes
            self._create_procedure_nodes(prog)

            # Create variable nodes
            self._create_variable_nodes(prog)

        # Create CALL relationships (after all programs are created)
        for prog in parsed_programs:
            self._create_call_relationships(prog)

        print("✓ Knowledge graph built successfully")

    def _create_program_node(self, prog: Dict[str, Any]):
        """Create CobolProgram node."""
        query = """
        MERGE (p:CobolProgram {name: $name})
        SET p.id = $name,
            p.author = $author,
            p.description = $description,
            p.code = $code
        """
        self.graph.query(query, {
            'name': prog['program_name'],
            'author': prog.get('author', 'Unknown'),
            'description': prog.get('description', ''),
            'code': prog['code']
        })

    def _create_file_relationships(self, prog: Dict[str, Any]):
        """Create DataFile nodes and relationships."""
        prog_name = prog['program_name']

        # Files declared
        for file_name in prog['files_declared']:
            query = """
            MERGE (f:DataFile {name: $file_name})
            SET f.id = $file_name
            WITH f
            MATCH (p:CobolProgram {name: $prog_name})
            MERGE (p)-[:DECLARES_FILE]->(f)
            """
            self.graph.query(query, {'file_name': file_name, 'prog_name': prog_name})

        # Files read
        for file_name in prog['files_read']:
            query = """
            MERGE (f:DataFile {name: $file_name})
            SET f.id = $file_name
            WITH f
            MATCH (p:CobolProgram {name: $prog_name})
            MERGE (p)-[:READS]->(f)
            """
            self.graph.query(query, {'file_name': file_name, 'prog_name': prog_name})

        # Files written
        for file_name in prog['files_written']:
            query = """
            MERGE (f:DataFile {name: $file_name})
            SET f.id = $file_name
            WITH f
            MATCH (p:CobolProgram {name: $prog_name})
            MERGE (p)-[:WRITES]->(f)
            """
            self.graph.query(query, {'file_name': file_name, 'prog_name': prog_name})

    def _create_procedure_nodes(self, prog: Dict[str, Any]):
        """Create Procedure nodes and relationships."""
        for proc_name in prog['procedures']:
            query = """
            MATCH (p:CobolProgram {name: $prog_name})
            MERGE (proc:Procedure {name: $proc_name, program: $prog_name})
            SET proc.id = $proc_name
            MERGE (p)-[:CONTAINS_PROCEDURE]->(proc)
            """
            self.graph.query(query, {
                'prog_name': prog['program_name'],
                'proc_name': proc_name
            })

    def _create_variable_nodes(self, prog: Dict[str, Any]):
        """Create Variable nodes and relationships."""
        for var_name in prog['variables'][:10]:  # Limit to avoid clutter
            query = """
            MATCH (p:CobolProgram {name: $prog_name})
            MERGE (v:Variable {name: $var_name, program: $prog_name})
            SET v.id = $var_name
            MERGE (p)-[:DECLARES_VARIABLE]->(v)
            """
            self.graph.query(query, {
                'prog_name': prog['program_name'],
                'var_name': var_name
            })

    def _create_call_relationships(self, prog: Dict[str, Any]):
        """Create CALLS relationships between programs."""
        for called_prog in prog['calls']:
            query = """
            MATCH (caller:CobolProgram {name: $caller})
            MATCH (called:CobolProgram {name: $called})
            MERGE (caller)-[:CALLS]->(called)
            """
            self.graph.query(query, {
                'caller': prog['program_name'],
                'called': called_prog
            })

# Build the knowledge graph
graph_builder = CobolKnowledgeGraphBuilder(graph)
graph_builder.build_graph(parsed_programs)

Building knowledge graph...
✓ Knowledge graph built successfully


In [94]:
# Verify the graph schema
graph.refresh_schema()
print("Knowledge Graph Schema:")
print("=" * 60)
print(graph.schema)

Knowledge Graph Schema:
Node properties:
CobolProgram {name: STRING, id: STRING, author: STRING, description: STRING, code: STRING}
DataFile {name: STRING, id: STRING}
Procedure {name: STRING, id: STRING, program: STRING}
Variable {name: STRING, id: STRING, program: STRING}
Relationship properties:

The relationships:
(:CobolProgram)-[:READS]->(:DataFile)
(:CobolProgram)-[:CONTAINS_PROCEDURE]->(:Procedure)
(:CobolProgram)-[:CALLS]->(:CobolProgram)
(:CobolProgram)-[:DECLARES_VARIABLE]->(:Variable)
(:CobolProgram)-[:WRITES]->(:DataFile)


In [95]:
# Quick stats on the graph
stats_query = """
MATCH (p:CobolProgram)
OPTIONAL MATCH (p)-[:CALLS]->(called)
OPTIONAL MATCH (p)-[:READS]->(f)
RETURN p.name AS Program,
       COUNT(DISTINCT called) AS Calls,
       COUNT(DISTINCT f) AS FilesRead
ORDER BY p.name
"""

stats = graph.query(stats_query)
print("\nProgram Statistics:")
print("=" * 60)
for row in stats:
    print(f"{row['Program']:15} | Calls: {row['Calls']} | Files Read: {row['FilesRead']}")


Program Statistics:
AUDITLOG        | Calls: 0 | Files Read: 0
CUSTADD         | Calls: 1 | Files Read: 0
CUSTMAST        | Calls: 3 | Files Read: 3
CUSTUPDT        | Calls: 1 | Files Read: 0
DAILYPROC       | Calls: 3 | Files Read: 0
REPORTGEN       | Calls: 1 | Files Read: 1
VALIDATE        | Calls: 0 | Files Read: 0


In [117]:
COBOL_CYPHER_PROMPT = PromptTemplate(
    input_variables=["schema", "question"],
    template="""You are a Neo4j expert analyzing COBOL legacy code modernization.

Task: Generate a Cypher query to answer the question about COBOL programs.

Schema:
{schema}

Query Patterns by Question Type:

1. DESCRIPTION queries ("what does", "describe", "tell me about"):
   MATCH (p:CobolProgram {{name: 'PROGRAM-NAME'}})
   OPTIONAL MATCH (p)-[:READS]->(rf:CobolFile)
   OPTIONAL MATCH (p)-[:WRITES]->(wf:CobolFile)
   OPTIONAL MATCH (p)-[:CALLS]->(called:CobolProgram)
   OPTIONAL MATCH (caller:CobolProgram)-[:CALLS]->(p)
   RETURN p.name as program_name,
          p.description as description,
          p.author as author,
          collect(DISTINCT rf.name) as reads_files,
          collect(DISTINCT wf.name) as writes_files,
          collect(DISTINCT called.name) as calls_programs,
          collect(DISTINCT caller.name) as called_by_programs

2. IMPACT ANALYSIS queries ("impact", "affect", "change", "modify"):
   MATCH (target:CobolProgram {{name: 'PROGRAM-NAME'}})

   // Direct upstream (programs that call target)
   OPTIONAL MATCH (caller1:CobolProgram)-[:CALLS]->(target)

   // Indirect upstream (2 hops)
   OPTIONAL MATCH (caller2:CobolProgram)-[:CALLS]->(:CobolProgram)-[:CALLS]->(target)
   WHERE NOT (caller2)-[:CALLS]->(target)

   // Direct downstream (programs target calls)
   OPTIONAL MATCH (target)-[:CALLS]->(dep1:CobolProgram)

   // Indirect downstream (2 hops)
   OPTIONAL MATCH (target)-[:CALLS]->(:CobolProgram)-[:CALLS]->(dep2:CobolProgram)
   WHERE NOT (target)-[:CALLS]->(dep2)

   // File coupling
   OPTIONAL MATCH (target)-[:READS|WRITES]->(file:CobolFile)<-[:READS|WRITES]-(coupled:CobolProgram)
   WHERE coupled.name <> target.name

   RETURN target.name as program,
          target.description as description,
          collect(DISTINCT caller1.name) as direct_callers,
          collect(DISTINCT caller2.name) as indirect_callers,
          collect(DISTINCT dep1.name) as direct_dependencies,
          collect(DISTINCT dep2.name) as indirect_dependencies,
          collect(DISTINCT {{program: coupled.name, file: file.name}}) as file_coupled

3. DEPENDENCY queries ("depends on", "uses", "calls"):
   MATCH (p:CobolProgram {{name: 'PROGRAM-NAME'}})
   OPTIONAL MATCH (p)-[:CALLS]->(dep:CobolProgram)
   OPTIONAL MATCH (p)-[:READS|WRITES]->(file:CobolFile)
   RETURN p.name,
          collect(DISTINCT dep.name) as program_dependencies,
          collect(DISTINCT file.name) as file_dependencies

4. CALLER queries ("who calls", "which programs call"):
   MATCH (target:CobolProgram {{name: 'PROGRAM-NAME'}})
   OPTIONAL MATCH (caller:CobolProgram)-[:CALLS]->(target)
   RETURN target.name,
          collect(DISTINCT caller.name) as callers

5. FILE USAGE queries ("which programs use file", "programs accessing"):
   MATCH (f:CobolFile {{name: 'FILE-NAME'}})
   OPTIONAL MATCH (reader:CobolProgram)-[:READS]->(f)
   OPTIONAL MATCH (writer:CobolProgram)-[:WRITES]->(f)
   RETURN f.name,
          collect(DISTINCT reader.name) as readers,
          collect(DISTINCT writer.name) as writers

Important Guidelines:
- Always use OPTIONAL MATCH to handle missing relationships
- Use collect(DISTINCT ...) to avoid duplicates
- For impact analysis, check BOTH directions: callers AND callees
- Split variable-length paths into separate hops to avoid length() errors
- Return descriptive field names
- Use WHERE NOT clauses to avoid duplicates in indirect matches

Question: {question}

Return ONLY the Cypher query without explanation or markdown:
"""
)
COBOL_QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a COBOL modernization expert analyzing legacy systems.

Question: {question}

Graph Database Results:
{context}

Instructions:

1. For DESCRIPTION questions (what does X do):
   - Explain the program's purpose based on its description
   - Mention what files it reads/writes
   - List programs it calls (downstream dependencies)
   - List programs that call it (upstream dependencies)
   - Keep it concise and clear

2. For IMPACT ANALYSIS questions (what happens if I change X):
   - Start with an impact summary
   - Categorize impacts:
     • CRITICAL: Programs that directly call the target (must test these)
     • MODERATE: Programs the target calls (coordination needed)
     • DATA COUPLING: Programs sharing the same files
   - Provide testing recommendations
   - Assess risk level (CRITICAL/HIGH/MEDIUM/LOW)

3. For DEPENDENCY questions (what does X depend on):
   - List direct program dependencies
   - List file dependencies
   - Explain the nature of each dependency

Format your response clearly with appropriate sections.
If the context is empty, state that no information was found in the knowledge graph.

Answer:
"""
)




In [118]:
# Create the Cypher QA chain with custom QA prompt
cypher_chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    verbose=True,
    cypher_prompt=COBOL_CYPHER_PROMPT,
    qa_prompt=COBOL_QA_PROMPT,
    return_intermediate_steps=False,
    allow_dangerous_requests=True,
)

print("✓ Cypher QA chain initialized with improved prompts")

✓ Cypher QA chain initialized with improved prompts


In [119]:
class SimpleCobolChatbot:
    """Simplified chatbot for COBOL legacy code queries using only Cypher (structural retrieval)."""

    def __init__(self, cypher_chain, graph):
        self.cypher_chain = cypher_chain
        self.graph = graph

    def query(self, question: str) -> Dict[str, Any]:
        """Process user question using only structural Cypher queries."""

        print(f"\n{'='*60}")
        print(f"Question: {question}")
        print(f"{'='*60}\n")
        print("🔗 Using Graph Cypher Query (Structural Retrieval Only)\n")

        try:
            # Use Cypher chain for all queries
            response = self.cypher_chain.invoke({"query": question})

            result = {
                'question': question,
                'answer': response['result'],
                'method': 'cypher',
                'cypher_query': response['intermediate_steps'][0]['query'] if response.get('intermediate_steps') else None,
                'context': response['intermediate_steps'][1]['context'] if response.get('intermediate_steps') else None,
                'evidence': self._extract_evidence(response)
            }

            self._display_response(result)
            return result

        except Exception as e:
            print(f"Error: {e}")
            return {'question': question, 'answer': f"Error processing query: {e}", 'error': str(e)}

    def _extract_evidence(self, response: Dict[str, Any]) -> List[Dict]:
        """Extract evidence and source traceability."""
        evidence = []

        if response.get('intermediate_steps'):
            context = response['intermediate_steps'][1].get('context', [])

            for item in context:
                if isinstance(item, dict):
                    # Extract program name and create evidence
                    prog_name = item.get('p.name') or item.get('caller.name') or item.get('Program') or item.get('ProgramName')
                    if prog_name:
                        evidence.append({
                            'type': 'graph_relationship',
                            'program': prog_name,
                            'data': item
                        })

        return evidence

    def _display_response(self, result: Dict[str, Any]):
        """Display formatted response with evidence."""

        print("\n📋 ANSWER:")
        print("-" * 60)
        print(result['answer'])

        if result.get('cypher_query'):
            print("\n🔍 GENERATED CYPHER QUERY:")
            print("-" * 60)
            print(result['cypher_query'])

        if result.get('evidence'):
            print("\n✓ EVIDENCE (Source Traceability):")
            print("-" * 60)
            for i, ev in enumerate(result['evidence'][:3], 1):
                if ev['type'] == 'graph_relationship':
                    print(f"{i}. Program: {ev['program']}")
                    print(f"   Data: {ev['data']}")

        print("\n" + "="*60)

    def get_program_code(self, program_name: str) -> str:
        """Retrieve source code for a program."""
        query = "MATCH (p:CobolProgram {name: $name}) RETURN p.code AS code"
        result = self.graph.query(query, {'name': program_name})
        return result[0]['code'] if result else "Program not found"

# Initialize simple chatbot (structural-only, no vector search)
simple_chatbot = SimpleCobolChatbot(cypher_chain, graph)
print("✓ Simple Chatbot initialized (Structural/Cypher retrieval only - no vector search)")

✓ Simple Chatbot initialized (Structural/Cypher retrieval only - no vector search)


In [120]:
# Test 1: Program information (re-run to see updated prompt effect)
simple_chatbot.query("What does the CUSTMAST program do?")


Question: What does the CUSTMAST program do?

🔗 Using Graph Cypher Query (Structural Retrieval Only)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:CobolProgram {name: 'CUSTMAST'})
OPTIONAL MATCH (p)-[:READS]->(rf:DataFile)
OPTIONAL MATCH (p)-[:WRITES]->(wf:DataFile)
OPTIONAL MATCH (p)-[:CALLS]->(called:CobolProgram)
OPTIONAL MATCH (caller:CobolProgram)-[:CALLS]->(p)
RETURN p.name as program_name,
       p.description as description,
       p.author as author,
       collect(DISTINCT rf.name) as reads_files,
       collect(DISTINCT wf.name) as writes_files,
       collect(DISTINCT called.name) as calls_programs,
       collect(DISTINCT caller.name) as called_by_programs
Full Context:
[{'program_name': 'CUSTMAST', 'description': 'CUSTOMER MASTER FILE MAINTENANCE PROGRAM                     * READS CUSTOMER TRANSACTIONS AND UPDATES MASTER FILE          *', 'author': 'LEGACY-TEAM.', 'reads_files': ['CUSTOMER', 'CALL', 'TRANS'], 'writes_files': [], 'calls_program

{'question': 'What does the CUSTMAST program do?',
 'answer': '**DESCRIPTION of CUSTMAST Program**\n\n- **Purpose**: The CUSTMAST program is designed for customer master file maintenance. It reads customer transactions and updates the master file accordingly.\n  \n- **Files Read**: \n  - CUSTOMER\n  - CALL\n  - TRANS\n\n- **Files Written**: None specified.\n\n- **Downstream Dependencies (Programs it calls)**:\n  - CUSTUPDT\n  - CUSTADD\n  - AUDITLOG\n\n- **Upstream Dependencies (Programs that call it)**:\n  - DAILYPROC\n\nThe CUSTMAST program is integral to maintaining the accuracy and currency of the customer master file by processing transaction data and coordinating with other programs for updates and logging.',
 'method': 'cypher',
 'cypher_query': None,
 'context': None,
 'evidence': []}

In [121]:
# Test 2: Basic program dependencies
simple_chatbot.query("Which programs does CUSTMAST call?")


Question: Which programs does CUSTMAST call?

🔗 Using Graph Cypher Query (Structural Retrieval Only)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:CobolProgram {name: 'CUSTMAST'})
OPTIONAL MATCH (p)-[:CALLS]->(dep:CobolProgram)
RETURN p.name,
       collect(DISTINCT dep.name) as called_programs
Full Context:
[{'p.name': 'CUSTMAST', 'called_programs': ['CUSTUPDT', 'CUSTADD', 'AUDITLOG']}]

> Finished chain.

📋 ANSWER:
------------------------------------------------------------
**Question: Which programs does CUSTMAST call?**

**Answer:**

Based on the graph database results, the program CUSTMAST calls the following programs:

1. **CUSTUPDT**
2. **CUSTADD**
3. **AUDITLOG**

These are the direct downstream dependencies of CUSTMAST. If you need further analysis on any of these programs, such as their purpose, the files they interact with, or their own dependencies, please provide additional context or specify the type of analysis required.



{'question': 'Which programs does CUSTMAST call?',
 'answer': '**Question: Which programs does CUSTMAST call?**\n\n**Answer:**\n\nBased on the graph database results, the program CUSTMAST calls the following programs:\n\n1. **CUSTUPDT**\n2. **CUSTADD**\n3. **AUDITLOG**\n\nThese are the direct downstream dependencies of CUSTMAST. If you need further analysis on any of these programs, such as their purpose, the files they interact with, or their own dependencies, please provide additional context or specify the type of analysis required.',
 'method': 'cypher',
 'cypher_query': None,
 'context': None,
 'evidence': []}

In [122]:
# Test 3: Impact analysis
simple_chatbot.query("If I modify the CUSTMAST program, which other programs will be affected?")


Question: If I modify the CUSTMAST program, which other programs will be affected?

🔗 Using Graph Cypher Query (Structural Retrieval Only)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (target:CobolProgram {name: 'CUSTMAST'})
OPTIONAL MATCH (caller1:CobolProgram)-[:CALLS]->(target)
OPTIONAL MATCH (caller2:CobolProgram)-[:CALLS]->(:CobolProgram)-[:CALLS]->(target)
WHERE NOT (caller2)-[:CALLS]->(target)
OPTIONAL MATCH (target)-[:CALLS]->(dep1:CobolProgram)
OPTIONAL MATCH (target)-[:CALLS]->(:CobolProgram)-[:CALLS]->(dep2:CobolProgram)
WHERE NOT (target)-[:CALLS]->(dep2)
OPTIONAL MATCH (target)-[:READS|WRITES]->(file:CobolFile)<-[:READS|WRITES]-(coupled:CobolProgram)
WHERE coupled.name <> target.name
RETURN target.name as program,
       target.description as description,
       collect(DISTINCT caller1.name) as direct_callers,
       collect(DISTINCT caller2.name) as indirect_callers,
       collect(DISTINCT dep1.name) as direct_dependencies,
       collect(DISTI

Full Context:
[{'program': 'CUSTMAST', 'description': 'CUSTOMER MASTER FILE MAINTENANCE PROGRAM                     * READS CUSTOMER TRANSACTIONS AND UPDATES MASTER FILE          *', 'direct_callers': ['DAILYPROC'], 'indirect_callers': [], 'direct_dependencies': ['CUSTUPDT', 'CUSTADD', 'AUDITLOG'], 'indirect_dependencies': ['VALIDATE'], 'file_coupled': [{'file': None, 'program': None}]}]

> Finished chain.

📋 ANSWER:
------------------------------------------------------------
**Impact Analysis for Modifying CUSTMAST Program**

**Impact Summary:**
Modifying the CUSTMAST program will have a direct impact on the programs that call it and those it calls. It is crucial to assess the potential effects on these related programs to ensure system stability and functionality.

**Categorized Impacts:**

- **CRITICAL:**
  - **DAILYPROC**: This program directly calls CUSTMAST. Any changes to CUSTMAST could affect the functionality of DAILYPROC, making it essential to thoroughly test this program a

{'question': 'If I modify the CUSTMAST program, which other programs will be affected?',
 'answer': '**Impact Analysis for Modifying CUSTMAST Program**\n\n**Impact Summary:**\nModifying the CUSTMAST program will have a direct impact on the programs that call it and those it calls. It is crucial to assess the potential effects on these related programs to ensure system stability and functionality.\n\n**Categorized Impacts:**\n\n- **CRITICAL:**\n  - **DAILYPROC**: This program directly calls CUSTMAST. Any changes to CUSTMAST could affect the functionality of DAILYPROC, making it essential to thoroughly test this program after modifications to CUSTMAST.\n\n- **MODERATE:**\n  - **CUSTUPDT**: CUSTMAST directly calls this program. Changes in CUSTMAST may require coordination with CUSTUPDT to ensure compatibility and functionality.\n  - **CUSTADD**: Similar to CUSTUPDT, this program is directly called by CUSTMAST, necessitating coordination and potential adjustments.\n  - **AUDITLOG**: As a d